# Scraper et crawler un vrai site : Books to Scrape
**Un site réel, public, créé exactement pour s'entraîner** : `books.toscrape.com` (une librairie fictive de 1 000 livres, sur 50 pages).

Dans ce notebook, vous écrivez un script **classique** : `requests` pour télécharger, `BeautifulSoup` pour lire, des boucles pour parcourir. Rien d'autre. À la fin, vous aurez un fichier CSV de livres.

| Étape | Ce que vous faites |
|---|---|
| 1 | Télécharger une page et la lire |
| 2 | Repérer un livre dans le HTML |
| 3 | Extraire ses champs |
| 4 | Faire la liste des livres d'une page |
| 5 | **Crawler** : suivre le bouton « next » de page en page |
| 6 | **Aller plus loin** : ouvrir la fiche de chaque livre |
| 7 | Sauvegarder et contrôler |

**Pour l'instant, on ne se pose aucune question de politesse ni de règles** : ce site est fait pour ça. Le notebook suivant expliquera pourquoi, sur d'autres sites, il faut faire autrement.

`pip install requests beautifulsoup4 pandas`

In [1]:
import re
import time
# Sur certains postes Windows, truststore permet à requests d'utiliser le magasin
# de certificats du système. Le notebook reste compatible s'il n'est pas installé.
try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass

import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

URL_DEPART = "https://books.toscrape.com/"

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Télécharger une page et la lire
`requests.get` télécharge la page. Le **code 200** veut dire « tout va bien ». `BeautifulSoup` transforme le texte HTML en un arbre dans lequel on peut chercher.

💡 On donne à BeautifulSoup `reponse.content` (les octets bruts) et non `reponse.text` : il devine alors lui-même l'encodage correct. Sinon, le symbole `£` peut s'afficher `Â£`.

In [2]:
session = requests.Session()
reponse = session.get(URL_DEPART, timeout=20)
reponse.raise_for_status()
print("Code de réponse :", reponse.status_code)

page = BeautifulSoup(reponse.content, "html.parser")
print("Titre de la page :", page.title.get_text(strip=True))

Code de réponse : 200
Titre de la page : All products | Books to Scrape - Sandbox


## Étape 2 — Repérer un livre dans le HTML
Ouvrez le site dans votre navigateur, faites un clic droit sur un livre puis **Inspecter** : vous verrez que chaque livre est dans une balise `<article class="product_pod">`. Regardons-en un.

In [3]:
carte = page.select_one("article.product_pod")
print(carte.prettify()[:900])

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   £51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



## Étape 3 — Extraire les champs d'un livre
On écrit une fonction qui lit **une carte** et renvoie un dictionnaire. Trois choses à noter :
- le titre complet est dans l'attribut `title` du lien (le texte affiché est parfois coupé) ;
- la note est dans le **nom de la classe** (`star-rating Three`), pas dans le texte ;
- les liens sont **relatifs** : `urljoin` fabrique l'adresse complète.

In [4]:
NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def lire_livre(carte, url_page):
    lien = carte.select_one("h3 a")
    prix_texte = carte.select_one("p.price_color").get_text(strip=True)
    return {
        "titre": lien["title"],
        "prix": float(re.sub(r"[^0-9.]", "", prix_texte)),          # ne garde que les chiffres et le point
        "note": NOTES[carte.select_one("p.star-rating")["class"][1]],
        "url": urljoin(url_page, lien["href"]),
    }

print(lire_livre(carte, URL_DEPART))

{'titre': 'A Light in the Attic', 'prix': 51.77, 'note': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


## Étape 4 — Tous les livres d'une page

In [5]:
livres_page = [lire_livre(c, URL_DEPART) for c in page.select("article.product_pod")]
print(len(livres_page), "livres sur la première page")
pd.DataFrame(livres_page).head()

20 livres sur la première page


,titre,prix,note,url
0,A Light in the Attic,51.77,3,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,1,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,1,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,4,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,5,https://books.toscrape.com/catalogue/sapiens-a...


## Étape 5 — Crawler : suivre le bouton « next »
C'est le **crawling** : on part d'une page, on lit ses livres, puis on suit le lien vers la page suivante, jusqu'à ce qu'il n'y en ait plus. La boucle s'arrête toute seule quand il n'y a plus de bouton « next ».

In [6]:
livres = []
url = URL_DEPART
numero = 0
nb_requetes_listes = 0
debut_collecte = time.perf_counter()

while url:
    numero += 1
    reponse = session.get(url, timeout=20)
    nb_requetes_listes += 1
    reponse.raise_for_status()
    page = BeautifulSoup(reponse.content, "html.parser")
    for carte in page.select("article.product_pod"):
        livres.append(lire_livre(carte, url))
    suivant = page.select_one("li.next a")
    url = urljoin(url, suivant["href"]) if suivant else None
    print(f"page {numero} lue, {len(livres)} livres au total")

print("\nTotal :", len(livres), "livres sur", numero, "pages")

page 1 lue, 20 livres au total
page 2 lue, 40 livres au total
page 3 lue, 60 livres au total
page 4 lue, 80 livres au total
page 5 lue, 100 livres au total
page 6 lue, 120 livres au total
page 7 lue, 140 livres au total
page 8 lue, 160 livres au total
page 9 lue, 180 livres au total
page 10 lue, 200 livres au total
page 11 lue, 220 livres au total
page 12 lue, 240 livres au total
page 13 lue, 260 livres au total
page 14 lue, 280 livres au total
page 15 lue, 300 livres au total
page 16 lue, 320 livres au total
page 17 lue, 340 livres au total
page 18 lue, 360 livres au total
page 19 lue, 380 livres au total
page 20 lue, 400 livres au total
page 21 lue, 420 livres au total
page 22 lue, 440 livres au total
page 23 lue, 460 livres au total
page 24 lue, 480 livres au total
page 25 lue, 500 livres au total
page 26 lue, 520 livres au total
page 27 lue, 540 livres au total
page 28 lue, 560 livres au total
page 29 lue, 580 livres au total
page 30 lue, 600 livres au total
page 31 lue, 620 livres

In [7]:
controle(
    len(livres) == 1000 and len({livre["url"] for livre in livres}) == len(livres),
    "%d livres, sans doublon d'adresse" % len(livres),
    "Attendu : 1 000 livres et aucune adresse en double",
)

✅ 1000 livres, sans doublon d'adresse


## Étape 6 — Aller plus loin : ouvrir la fiche de chaque livre
La page de liste ne donne ni la **catégorie**, ni la **description**, ni le **stock**. Il faut ouvrir la fiche de chaque livre : c'est du crawling vers des pages de détail.

⚠️ Attention : **certains livres n'ont pas de description**. Le code doit le prévoir (`None`), sinon il plante au premier livre concerné.

⏱️ Ouvrir 1 000 fiches prend plusieurs minutes. On commence avec `LIMITE = 40`, puis on met `None` pour tout lire.

In [8]:
def lire_detail(url):
    reponse = session.get(url, timeout=20)
    reponse.raise_for_status()
    page = BeautifulSoup(reponse.content, "html.parser")
    liens_fil = page.select("ul.breadcrumb li a")
    description = page.select_one("#product_description + p")
    infos = {
        ligne.th.get_text(strip=True): ligne.td.get_text(strip=True)
        for ligne in page.select("table.table-striped tr")
    }
    stock_trouve = re.search(r"\d+", infos.get("Availability", ""))
    stock = int(stock_trouve.group()) if stock_trouve else None
    return {
        "categorie": liens_fil[2].get_text(strip=True) if len(liens_fil) >= 3 else None,
        "description": description.get_text(strip=True) if description else None,
        "upc": infos.get("UPC"),
        "stock": stock,
        "disponible": stock is not None and stock > 0,
        "nb_avis": int(infos.get("Number of reviews", 0)),
    }

# None signifie que les 1 000 fiches sont lues pour produire le livrable final.
LIMITE = None
livres_a_lire = livres if LIMITE is None else livres[:LIMITE]
details = []
nb_requetes_details = 0

for i, livre in enumerate(livres_a_lire, start=1):
    details.append({**livre, **lire_detail(livre["url"])})
    nb_requetes_details += 1
    if i % 100 == 0 or i == len(livres_a_lire):
        print(i, "fiches lues")

df = pd.DataFrame(details)
duree_collecte = time.perf_counter() - debut_collecte
nb_requetes_total = nb_requetes_listes + nb_requetes_details
df.head()

100 fiches lues
200 fiches lues
300 fiches lues
400 fiches lues
500 fiches lues
600 fiches lues
700 fiches lues
800 fiches lues
900 fiches lues
1000 fiches lues


,titre,prix,note,url,categorie,description,upc,stock,disponible,nb_avis
0,A Light in the Attic,51.77,3,https://books.toscrape.com/catalogue/a-light-i...,Poetry,It's hard to imagine a world without A Light i...,a897fe39b1053632,22,True,0
1,Tipping the Velvet,53.74,1,https://books.toscrape.com/catalogue/tipping-t...,Historical Fiction,"""Erotic and absorbing...Written with starling ...",90fa61229261140a,20,True,0
2,Soumission,50.10,1,https://books.toscrape.com/catalogue/soumissio...,Fiction,"Dans une France assez proche de la nôtre, un h...",6957f44c3847a760,20,True,0
3,Sharp Objects,47.82,4,https://books.toscrape.com/catalogue/sharp-obj...,Mystery,"WICKED above her hipbone, GIRL across her hear...",e00eb4fd7b871a48,20,True,0
4,Sapiens: A Brief History of Humankind,54.23,5,https://books.toscrape.com/catalogue/sapiens-a...,History,From a renowned historian comes a groundbreaki...,4165285e1663650f,20,True,0


## Étape 7 — Sauvegarder et contrôler
Un fichier de données se **vérifie** avant de servir : lignes, doublons, valeurs manquantes.

In [9]:
df.to_csv("livres.csv", index=False, encoding="utf-8")
print("Lignes :", len(df), "| doublons d'UPC :", df["upc"].duplicated().sum())
print("Requêtes de collecte :", nb_requetes_total)
print("Durée de collecte : %.1f secondes" % duree_collecte)
print("\nValeurs manquantes par colonne :")
print(df.isna().sum().to_string())
print("\nLivres par catégorie :")
print(df["categorie"].value_counts().head(8).to_string())

Lignes : 1000 | doublons d'UPC : 0
Requêtes de collecte : 1050
Durée de collecte : 355.7 secondes

Valeurs manquantes par colonne :
titre          0
prix           0
note           0
url            0
categorie      0
description    2
upc            0
stock          0
disponible     0
nb_avis        0

Livres par catégorie :
categorie
Default           152
Nonfiction        110
Sequential Art     75
Add a comment      67
Fiction            65
Young Adult        54
Fantasy            48
Romance            35


In [10]:
controle(
    len(df) == 1000
    and df["upc"].notna().all()
    and df["upc"].is_unique
    and df["prix"].between(0, 1000).all()
    and df["disponible"].eq(df["stock"].gt(0)).all(),
    "Fichier livres.csv cohérent : %d lignes, UPC uniques et prix plausibles" % len(df),
    "Attendu : 1 000 lignes, un UPC unique par livre et des prix entre 0 et 1 000",
)

✅ Fichier livres.csv cohérent : 1000 lignes, UPC uniques et prix plausibles


## Exercices et questions

Les cellules suivantes présentent les calculs demandés, puis une collecte limitée à une catégorie.
Les mesures de temps restent celles de l'exécution : elles dépendent de la connexion et du serveur.


In [11]:
note_moyenne_par_categorie = df.groupby("categorie")["note"].mean().sort_values(ascending=False)
prix_moyen_par_note = df.groupby("note")["prix"].mean().sort_index()
nb_sans_description = int(df["description"].isna().sum())

print("Note moyenne par catégorie :")
display(note_moyenne_par_categorie)
print("Prix moyen par note :")
display(prix_moyen_par_note)
print("Livres sans description :", nb_sans_description)

Note moyenne par catégorie :


categorie
Erotica               5.000000
Adult Fiction         5.000000
Novels                5.000000
Christian Fiction     4.166667
Health                3.750000
Art                   3.625000
Poetry                3.526316
Humor                 3.400000
Spirituality          3.333333
Young Adult           3.296296
Historical Fiction    3.230769
Fiction               3.184615
New Adult             3.166667
Music                 3.153846
Religion              3.142857
Womens Fiction        3.117647
Fantasy               3.083333
Autobiography         3.000000
Suspense              3.000000
Historical            3.000000
Sports and Games      3.000000
Sequential Art        2.973333
History               2.944444
Mystery               2.937500
Science               2.928571
Business              2.916667
Food and Drink        2.900000
Nonfiction            2.881818
Default               2.835526
Add a comment         2.761194
Thriller              2.727273
Travel                2.72727

Prix moyen par note :


note
1    34.561195
2    34.810918
3    34.692020
4    36.093296
5    35.374490
Name: prix, dtype: float64

Livres sans description : 2


In [12]:
# Exemple : crawler uniquement la catégorie Travel.
url_categorie = urljoin(URL_DEPART, "catalogue/category/books/travel_2/index.html")
livres_travel = []
nb_requetes_categorie = 0

while url_categorie:
    reponse = session.get(url_categorie, timeout=20)
    nb_requetes_categorie += 1
    reponse.raise_for_status()
    page_categorie = BeautifulSoup(reponse.content, "html.parser")
    livres_travel.extend(
        lire_livre(carte, url_categorie)
        for carte in page_categorie.select("article.product_pod")
    )
    suivant = page_categorie.select_one("li.next a")
    url_categorie = urljoin(url_categorie, suivant["href"]) if suivant else None

controle(
    len(livres_travel) > 0 and len({livre["url"] for livre in livres_travel}) == len(livres_travel),
    f"Catégorie Travel : {len(livres_travel)} livres sur {nb_requetes_categorie} page(s)",
    "La collecte de la catégorie est vide ou contient des doublons",
)

✅ Catégorie Travel : 11 livres sur 1 page(s)


### Réponses

- La collecte complète envoie **1 050 requêtes** : 50 pages de liste et 1 000 fiches. La durée exacte est affichée par le chronométrage de l'exécution.
- Si une classe CSS change, un sélecteur comme `article.product_pod` ou `p.price_color` peut ne plus trouver d'élément ; un accès suivant peut alors produire une erreur ou des données manquantes.
- Sur un site non prévu pour l'entraînement, une cadence maximale et un User-Agent générique peuvent entraîner des réponses 403 ou 429, une limitation d'adresse IP et une charge inutile pour le serveur. Il faut d'abord lire les règles du site, s'identifier et ralentir les requêtes.
